In [32]:
import pandas as pd
import numpy as np

#Input
student_id = 20250186   # CHANGE THIS
file_path = r"C:\Users\Ram Cedric\OneDrive\Documents\Lab6_Additional_Dataset\Most Popular Programming Languages.csv"

df = pd.read_csv(file_path)

# Convert Month to datetime
df['Month'] = pd.to_datetime(df['Month'])

#sort
languages = list(df.columns[1:])  # exclude Month
languages.sort()

print("Languages (Alphabetical):")
print(languages)

#Assign
last_three = int(str(student_id)[-3:])
index = last_three % len(languages)

assigned_language = languages[index]
print(f"\nAssigned Language: {assigned_language}")

#Extract
data = df[['Month', assigned_language]].copy()
data.columns = ['Month', 'Popularity']

Languages (Alphabetical):
['C# Worldwide(%)', 'Flutter Worldwide(%)', 'Java Worldwide(%)', 'JavaScript Worldwide(%)', 'Matlab Worldwide(%)', 'PhP Worldwide(%)', 'Python Worldwide(%)', 'React Worldwide(%)', 'Swift Worldwide(%)', 'TypeScript Worldwide(%)']

Assigned Language: Python Worldwide(%)


In [33]:
#1. Programming Language Growth Analysis
# Growth Rate
data['Growth_Rate'] = data['Popularity'].pct_change() * 100

# Rolling stats (12 months)
data['Moving_Avg'] = data['Popularity'].rolling(window=12).mean()
data['Moving_STD'] = data['Popularity'].rolling(window=12).std()

# Phase Classification
conditions = [
    data['Growth_Rate'] > 5,
    data['Growth_Rate'] < -5
]

choices = ['Growth', 'Decline']

data['Phase'] = np.select(conditions, choices, default='Stable')

# Stats
stats_summary = data['Popularity'].describe()

# Phase counts
phase_counts = data['Phase'].value_counts()

# Overall Growth
initial_value = data['Popularity'].iloc[0]
final_value = data['Popularity'].iloc[-1]
overall_growth = ((final_value - initial_value) / initial_value) * 100

#Display
print("1. Programming Language Growth Analysis")
print(f"Assigned Language: {assigned_language}\n")
print(data[['Month','Popularity','Growth_Rate','Moving_Avg','Moving_STD','Phase']].head())
print("\nStatistical Summary:\n", stats_summary)
print("\nPhase Counts:\n", phase_counts)

print(f"\nInitial Value: {initial_value}")
print(f"Final Value: {final_value}")
print(f"Overall Growth (%): {overall_growth:.2f}")

1. Programming Language Growth Analysis
Assigned Language: Python Worldwide(%)

       Month  Popularity  Growth_Rate  Moving_Avg  Moving_STD   Phase
0 2004-01-01          30          NaN         NaN         NaN  Stable
1 2004-02-01          29    -3.333333         NaN         NaN  Stable
2 2004-03-01          28    -3.448276         NaN         NaN  Stable
3 2004-04-01          28     0.000000         NaN         NaN  Stable
4 2004-05-01          28     0.000000         NaN         NaN  Stable

Statistical Summary:
 count    249.000000
mean      41.678715
std       23.103231
min       20.000000
25%       23.000000
50%       29.000000
75%       60.000000
max      100.000000
Name: Popularity, dtype: float64

Phase Counts:
 Phase
Stable     162
Growth      50
Decline     37
Name: count, dtype: int64

Initial Value: 30
Final Value: 79
Overall Growth (%): 163.33


In [34]:
#2. Lifecycle Classification of Programming Languages
print("2. Lifecycle Classification of Programming Language")

#Growth rate (%)
data['Growth_Rate'] = data['Popularity'].pct_change() * 100

# Rolling (6 months)
data['Moving_Avg'] = data['Popularity'].rolling(window=6).mean()
data['Moving_STD'] = data['Popularity'].rolling(window=6).std()

# Mean and Std
mean_growth = np.mean(data['Growth_Rate'])
std_growth = np.std(data['Growth_Rate'])

#Conditions
conditions = [
    (data['Growth_Rate'] > 0) & (data['Growth_Rate'] < mean_growth),
    (data['Growth_Rate'] > mean_growth),
    (abs(data['Growth_Rate']) <= 1),
    (data['Growth_Rate'] < -std_growth)
]

choices = ['Introduction', 'Growth', 'Maturity', 'Decline']

data['Lifecycle_Phase'] = np.select(conditions, choices, default='Maturity')

#Table
print(data[['Month', 'Popularity', 'Growth_Rate', 'Moving_Avg', 'Moving_STD', 'Lifecycle_Phase']].head())

life_counts = data['Lifecycle_Phase'].value_counts()
total = len(data)

#Display
print("\nLifecycle Counts:")
print(life_counts)

print("\nPercentages:")
print((life_counts / life_counts.sum()) * 100) 

# Dominant Stage
dominant_stage = life_counts.idxmax()

print(f"\nDominant Stage: {dominant_stage}")

2. Lifecycle Classification of Programming Language
       Month  Popularity  Growth_Rate  Moving_Avg  Moving_STD Lifecycle_Phase
0 2004-01-01          30          NaN         NaN         NaN        Maturity
1 2004-02-01          29    -3.333333         NaN         NaN        Maturity
2 2004-03-01          28    -3.448276         NaN         NaN        Maturity
3 2004-04-01          28     0.000000         NaN         NaN        Maturity
4 2004-05-01          28     0.000000         NaN         NaN        Maturity

Lifecycle Counts:
Lifecycle_Phase
Maturity    123
Growth      100
Decline      26
Name: count, dtype: int64

Percentages:
Lifecycle_Phase
Maturity    49.397590
Growth      40.160643
Decline     10.441767
Name: count, dtype: float64

Dominant Stage: Maturity


In [35]:
#3. Lifecycle Classification of Programming Languages 
print("\n3. Comparative Analysis of Programming Languages\n")

first_lang = assigned_language
second_index = (index + 1) % len(languages)
second_lang = languages[second_index]

print(f"First Language (A): {first_lang}")
print(f"Second Language (B): {second_lang}\n")

comp = df[['Month', first_lang, second_lang]].copy()
comp.columns = ['Month', 'A', 'B']

# Stats
print("Mean A:", comp['A'].mean())
print("Mean B:", comp['B'].mean())

print("Median A:", comp['A'].median())
print("Median B:", comp['B'].median())

print("Std A:", comp['A'].std())
print("Std B:", comp['B'].std())

# Correlation
corr = np.corrcoef(comp['A'], comp['B'])[0,1]
print("\nCorrelation:", corr)

# Dominance
dominance = np.sum(comp['A'] > comp['B']) / len(comp) * 100
print("Dominance Ratio (%):", dominance)

# RPI
rpi = comp['A'].mean() - comp['B'].mean()
print("Relative Performance Index (RPI):", rpi)

# Crossovers
comp['Compare'] = np.where(comp['A'] > comp['B'], 1, 0)
comp['Cross'] = comp['Compare'].diff()
crossovers = comp[comp['Cross'] == 1]
print("\nCross-over Months:")
print(crossovers['Month'].head(10))

# Summary Table
summary = pd.DataFrame({
    "Mean_A":[comp['A'].mean()],
    "Mean_B":[comp['B'].mean()],
    "Std_A":[comp['A'].std()],
    "Std_B":[comp['B'].std()],
    "Correlation":[corr],
    "Dominance_Ratio":[dominance],
    "RPI":[rpi]
})

print("\nComparative Summary:")
print(summary)


3. Comparative Analysis of Programming Languages

First Language (A): Python Worldwide(%)
Second Language (B): React Worldwide(%)

Mean A: 41.678714859437754
Mean B: 25.883534136546185
Median A: 29.0
Median B: 2.0
Std A: 23.10323057188936
Std B: 32.51820969323286

Correlation: 0.9865943613955088
Dominance Ratio (%): 89.5582329317269
Relative Performance Index (RPI): 15.795180722891569

Cross-over Months:
188   2019-09-01
224   2022-09-01
229   2023-02-01
237   2023-10-01
Name: Month, dtype: datetime64[us]

Comparative Summary:
      Mean_A     Mean_B      Std_A     Std_B  Correlation  Dominance_Ratio  \
0  41.678715  25.883534  23.103231  32.51821     0.986594        89.558233   

         RPI  
0  15.795181  
